In [0]:
import requests

import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# 🔐 GET FROM DATABRICKS SECRETS (FIX)

BASE_URL = dbutils.secrets.get(scope="slainte-secrets", key="supabase-url").strip()

API_KEY  = dbutils.secrets.get(scope="slainte-secrets", key="supabase-api-key").strip()

HEADERS  = {"apikey": API_KEY, "Authorization": f"Bearer {API_KEY}"}

# Fetch

def fetch(table):

    df = pd.DataFrame(requests.get(f"{BASE_URL}/{table}", headers=HEADERS).json())

    return df.where(lambda x: x.notna(), None)

pr = fetch("penalty_rules")

ji = fetch("jira_integrations")

# Explicit columns

pr = pr[[

    "id", "integration_id", "project",

    "priority_group", "priority_group_name", "priority_group_labels",

    "penalty_code", "penalty_type", "penalty_value", "penalty_unit",

    "condition", "status", "created_at", "updated_at", "project_key"

]]

# To Spark

df_pr = spark.createDataFrame(pr).withColumn("integration_id", F.col("integration_id").cast(StringType()))

df_ji = spark.createDataFrame(ji).withColumn("id", F.col("id").cast(StringType()))

# Join to get user_id

df = df_pr.join(

    df_ji.select(F.col("id").alias("integration_id"), "user_id"),

    on="integration_id",

    how="left"

).withColumn("ingestion_timestamp", F.current_timestamp())

# Overwrite

TARGET = "workspace.slainte_bronze.penalty_rules_bronze"

df.write.mode("overwrite").format("delta").saveAsTable(TARGET)

print("✅ Saved:", TARGET)

print("Rows:", df.count())
 